# EchoTTS · train on a cloud GPU

Works on **Google Colab** (Runtime → Change runtime type → GPU) and **Kaggle** (Accelerator → GPU, Internet on). Checkpoints go to persistent storage, so when the session is cut off you just run all cells again and training resumes from the newest checkpoint.

Run cells top to bottom. Only run ONE of the two training cells at a time.

In [ ]:
!nvidia-smi

## 1. Persistent storage
Colab → Google Drive. Kaggle → `/kaggle/working` (save the notebook version with *Save output* to keep it, or upload checkpoints to a Kaggle dataset).

In [ ]:
import os, pathlib
ON_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')
ON_KAGGLE = os.path.exists('/kaggle')
if ON_KAGGLE:
    PERSIST = pathlib.Path('/kaggle/working/echotts_store'); WORK = pathlib.Path('/kaggle/temp')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    PERSIST = pathlib.Path('/content/drive/MyDrive/echotts'); WORK = pathlib.Path('/content')
PERSIST.mkdir(parents=True, exist_ok=True); WORK.mkdir(parents=True, exist_ok=True)
print('persistent storage:', PERSIST)

## 2. Code

In [ ]:
REPO_DIR = WORK / 'echotts'
if REPO_DIR.exists():
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/isaad-ui/echotts.git {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt
!pytest -q

## 3. LJSpeech
On Kaggle you can instead *Add data* → search **LJSpeech** and it is used automatically.

In [ ]:
import glob
attached = glob.glob('/kaggle/input/**/LJSpeech-1.1', recursive=True)
if attached:
    DATA = pathlib.Path(attached[0])
else:
    DATA = WORK / 'data' / 'LJSpeech-1.1'
    if not (DATA / 'metadata.csv').exists():
        (WORK / 'data').mkdir(exist_ok=True)
        !wget -q -O {WORK}/data/LJSpeech-1.1.tar.bz2 https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2
        !tar -xjf {WORK}/data/LJSpeech-1.1.tar.bz2 -C {WORK}/data && rm {WORK}/data/LJSpeech-1.1.tar.bz2
print(DATA, len(list((DATA / 'wavs').glob('*.wav'))), 'wavs')

## 4. Precompute mels (once)
Mels are stored in persistent storage and linked into the data folder, so later sessions skip this step. Read-only inputs (Kaggle) get a writable copy of metadata + a link to the wavs.

In [ ]:
import shutil
MEL_STORE = PERSIST / 'mels'; MEL_STORE.mkdir(exist_ok=True)
TRAIN_DATA = WORK / 'ljs_view'
TRAIN_DATA.mkdir(exist_ok=True)
for name, target in [('wavs', DATA / 'wavs'), ('mels', MEL_STORE)]:
    link = TRAIN_DATA / name
    if not link.exists(): link.symlink_to(target, target_is_directory=True)
shutil.copy(DATA / 'metadata.csv', TRAIN_DATA / 'metadata.csv')
!python -m echotts.precompute_mels --data_dir {TRAIN_DATA}
print(len(list(MEL_STORE.glob('*.npy'))), 'mels ready')

## 5. TensorBoard
Watch `train/total` fall in the first 500 steps and check the alignment images every few thousand steps.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {PERSIST}/runs

## 5b. Overfit check (run this BEFORE training)
Trains on 8 fixed clips for 300 steps (a few minutes). The pictures appear right under the cell. **Pass** = the loss falls far below its start AND the attention picture shows a clear bright diagonal line from bottom-left to top-right. No diagonal → stop and debug before spending GPU hours.

In [ ]:
from IPython.display import Image, display
OVERFIT = PERSIST / 'overfit'
!python -m echotts.overfit --data_dir {TRAIN_DATA} --out_dir {OVERFIT}
for name in ['loss_curve.png', 'attention.png', 'mel_compare.png']:
    path = OVERFIT / name
    if path.exists():
        print(name); display(Image(str(path)))
    else:
        print('missing', name, '- the run failed, read the error above')

## 6a. Train the acoustic model
Resumes automatically. `--amp` = mixed precision (faster on T4). Change `--batch_size 32` to 16 if you run out of GPU memory. The folder name `acoustic_r2` matches `frames_per_step: 2` in the config: if you ever change that setting, use a new folder.

In [ ]:
!python -m echotts.train_acoustic --data_dir {TRAIN_DATA} --amp --batch_size 32 \
    --checkpoint_dir {PERSIST}/checkpoints/acoustic_r2 --log_dir {PERSIST}/runs/acoustic_r2

## 6b. Train the vocoder
Trains on real recordings, independent of the acoustic model, so it can run in a separate session (or a second free GPU account) in parallel.

In [ ]:
!python -m echotts.train_vocoder --data_dir {TRAIN_DATA} \
    --checkpoint_dir {PERSIST}/checkpoints/vocoder --log_dir {PERSIST}/runs/vocoder